In [9]:
# conda activate genomic_tools

import os
import sys
import pysam
import pickle
import pandas as pd
from Bio.Seq import Seq
from pyfaidx import Fasta
from collections import defaultdict

sys.path.append("code")

from modified_functions import *

pd.set_option('display.max_colwidth', None)

In [2]:
# Sequence-based domain detection (run HMMER/ELM directly on the exon peptide) sidesteps coordinates — a match is a match regardless of numbering. 
# But position-based cross-referencing (UniProt features, mapping full-protein InterProScan hits onto exons, isoform diffs) genuinely needs a correct, shared Met1.

In [4]:
def score_transcript(t, x):
    tag = str(x['transcript_tag'])
    return (
        int("MANE_Select" in tag),
        int("MANE_Plus_Clinical" in tag),
        int("appris_principal" in tag),
        int("basic" in tag),
        int("CCDS" in tag),
        int("GENCODE_Primary" in tag),
        x['coding_nt_length'] if "coding_nt_length" in x.keys() else 0,    # prefer longer (more complete) CDS
        t   # deterministic alphabetical tiebreak
    ) 

def _cds_rows(obj):
    if obj is None:
        return None
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end']), 'frame': int(r['frame'])}
                for _, r in obj.iterrows()]
    return [{'start': int(c['start']), 'end': int(c['end']), 'frame': int(c['frame'])} for c in obj]
 
def _make_protein_sequence(transcript, exon_dict, rec, cds_by_transcript, genome, skip=False):
    cds = cds_by_transcript.get(transcript)
    if cds is None:
        return None

    chrom = rec['meta'].get("chrom")
    strand = rec['meta'].get("strand")
    coding_seq = ''
    for c in _cds_rows(cds):
        if skip and c['start'] == exon_dict.get('exon_cds_start') \
               and c['end'] == exon_dict.get('exon_cds_end'):
            continue
        seq = genome.fetch(chrom, c['start'] - 1, c['end'])
        if strand == '-':
            seq = str(Seq(seq).reverse_complement())
        coding_seq += seq

    if not coding_seq:
        return None
    protein = str(Seq(coding_seq).translate(to_stop=True))
    return protein if protein else None

def get_coding_nt_length(transcript, cds_by_transcript):
    cds = cds_by_transcript.get(transcript)
    if cds is None:
        return 0
    return sum(c['end'] - c['start'] + 1 for c in _cds_rows(cds))

In [6]:
event_dicts = pickle.load(open('data/event_dicts.pkl', 'rb'))
event_info = pickle.load(open('data/event_info.pkl', 'rb'))

In [5]:
# Get all significant splicing events
signif_events = []
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        for idx, _ in signif_exons_df.iterrows():
            if idx not in signif_events:
                signif_events.append(idx)

In [7]:
genome_fasta = pysam.FastaFile("/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/GRCh38.primary_assembly.genome.fa") 

In [8]:
# Load GTF

exclude = ""
gene_name = "gene_name"
gene_type = "all"
no_trim_id = False
gene_type_tag = "gene_type"
transcript_type_tag = "transcript_type"

gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v46.annotation.gtf"
gtf = process_gtf(gtf_file, exclude, gene_name, no_trim_id, gene_type_tag, transcript_type_tag)

gtf_cds = gtf[gtf.feature == "CDS"]
cds_by_transcript = {t: grp for t, grp in gtf_cds.groupby('transcript')}  # for transcript coding sequences lookup

gtf_exon = gtf[gtf.feature == "exon"]
exons_by_transcript = {t: grp for t, grp in gtf_exon.groupby('transcript')}  # for transcript lookup

Processing GTF file...


INFO:root:Extracted GTF attributes: ['gene_id', 'gene_type', 'gene_name', 'level', 'tag', 'transcript_id', 'transcript_type', 'transcript_name', 'transcript_support_level', 'havana_transcript', 'exon_number', 'exon_id', 'hgnc_id', 'havana_gene', 'ont', 'protein_id', 'ccdsid', 'artif_dupl']


In [10]:
proteins = Fasta("/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v49.pc_translations.fa")

# Build a dict keyed by ENST
protein_by_transcript = {}
for key in proteins.keys():
    parts = key.split("|")
    enst_versioned = parts[1].split(".")[0]
    protein_by_transcript[enst_versioned] = str(proteins[key])

In [ ]:
event_by_coords = {(ed['es'], ed['ee']): ed['event'] for ed in event_dicts}

MIN_AA = 30
        
interproscan_targets = {}
event_protein_map  = {}

for ev, rec in event_info.items():
    if ev not in signif_events:
        continue

    # --- inclusion: best compatible protein-coding transcript per signif. event
    comp = {t: d for t, d in rec['compatible'].items()
            if isinstance(d.get('overlap_type'), str)
            and d['overlap_type'] != 'noncoding_or_utr'
            and d.get('transcript_type') == 'protein_coding'}
    if not comp:
        continue
    
    best_t = max(comp, key=lambda t: score_transcript(t, comp[t]))
    d = comp[best_t]
    chrom = rec['meta']['chrom']

    event_protein_map[ev] = {
        'inclusion': best_t,
        'aa_start': d['aa_start'],
        'aa_end': d['aa_end'],
        'frame_preserving': d['frame_preserving'],
        'clean_start': d['clean_start'],
        'clean_end': d['clean_end'],
        'real_skip': None,
        'siblings': []
    }

    # --- real skipped exon transcripts
    skip_candidates = {}
    for skip_t in rec['exon_skipped']:
        exons = exons_by_transcript.get(skip_t)
        if exons is None:
            continue
        ttype = exons.iloc[0].get('transcript_type', '')
        tag = exons.iloc[0].get('tag', '')
        if ttype != 'protein_coding':
            continue
        skip_candidates[skip_t] = {
            'transcript_tag':   tag,
            'coding_nt_length': get_coding_nt_length(skip_t, cds_by_transcript)
        }
            
    if skip_candidates:
        best_skip_t = max(skip_candidates, key=lambda t: score_transcript(t, skip_candidates[t]))
        event_protein_map[ev]['real_skip'] = best_skip_t
    
    # --- synthetic skipped exon transcripts: same backbone with cassette exon excised
    skip_seq = _make_protein_sequence(best_t, d, rec, cds_by_transcript, genome_fasta, skip=True)
    if skip_seq:
        if len(skip_seq) >= MIN_AA:
            skip_id = f"{ev}_synthetic_skip"
            interproscan_targets[skip_id] = skip_seq
            event_protein_map[ev]['synthetic_skip'] = skip_id
        elif not d['frame_preserving']:
            # frame-shifting: record truncation point without InterProScan
            event_protein_map[ev]['synthetic_skip'] = None
        event_protein_map[ev]['truncation_aa'] = d['aa_start']
        
    # --- siblings: detected boundary variants
    for sib_t, sib in rec['exon_diff_boundary'].items():
        if not sib.get('is_called_sibling'):
            continue
        sib_ev = event_by_coords.get((sib['start'], sib['end']))
        if not sib_ev or sib_ev not in event_info:
            continue
        sib_comp = {t: d for t, d in event_info[sib_ev]['compatible'].items()
                    if isinstance(d.get('overlap_type'), str)
                    and d['overlap_type'] != 'noncoding_or_utr'
                    and d.get('transcript_type') == 'protein_coding'}
        if not sib_comp:
            continue
        best_sib_t = max(sib_comp, key=lambda t: score_transcript(t, sib_comp[t]))
        event_protein_map[ev]['siblings'].append(best_sib_t)

/mnt/lareaulab/reliscu/anaconda3/envs/genomic_tools/lib/python3.14/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/mnt/lareaulab/reliscu/anaconda3/envs/genomic_tools/lib/python3.14/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/mnt/lareaulab/reliscu/anaconda3/envs/genomic_tools/lib/python3.14/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/mnt/lareaulab/reliscu/anaconda3/envs/genomic_tools/lib/python3.14/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon,

In [42]:
len(event_protein_map)

13966

In [52]:
transcripts = set()
for info in event_protein_map.values():
    if info.get('inclusion'):
        transcripts.add(info['inclusion'])
    if info.get('real_skip'):
        transcripts.add(info['real_skip'])
    transcripts.update(info.get('siblings', []))
    transcripts.update(info.get('real_skips', []))

In [53]:
len(transcripts)

17998

In [54]:
len(interproscan_targets)

12622

In [55]:
# write protein sequences

modified_transcript_products = {}
transcript_log = []

with open("data/proteins.fa", "w") as f:
    # for real transcripts:
    for key in proteins.keys():
        enst = key.split("|")[1].split(".")[0]
        if enst in transcripts:
            transcript_log.append(enst)
            seq = str(proteins[key]).rstrip("*")
            if "X" in seq:
                # track where in the sequence the X was 
                modified_transcript_products[enst] = [i for i, c in enumerate(seq) if c == "X"]
                seq = seq.replace("X", "")
            f.write(f">{enst}\n{seq}\n")
    # for synthetic transcripts:
    for id, seq in interproscan_targets.items():
        f.write(f">{id}\n{seq}\n")

# Ignore

## Goal: narrow down the space of transcripts being considered for runningw ith InterProScan

### First load in gene descriptions from Ensembl and Uniprot. 
In addition to selecting exons with the most significant cell type associations, I want to prioritize exons from genes with certain function, e.g. receptors, ion channels, synaptic proteins, etc.

In [3]:
ensembl_info = pd.read_csv("data/gene_descriptions.csv")
uniprot_info = pd.read_csv("data/gene_uniprot_info.csv")

In [4]:
gene_info = uniprot_info.merge(ensembl_info, on="gene", how="outer")

In [5]:
gene_info.head(20)

,gene,protein_name,function,keywords,description
0,A1BG,Alpha-1B-glycoprotein,NaN,"Alternative splicing, Direct protein sequencing, Disulfide bond, Glycoprotein, Immunoglobulin domain, Proteomics identification, Reference proteome, Repeat, Secreted, Signal",alpha-1-B glycoprotein [Source:HGNC Symbol;Acc:HGNC:5]
1,A1BG-AS1,NaN,NaN,NaN,A1BG antisense RNA 1 [Source:HGNC Symbol;Acc:HGNC:37133]
2,AAAS,Aladin,"Plays a role in the normal development of the peripheral and central nervous system (PubMed:11062474, PubMed:11159947, PubMed:16022285). Required for the correct localization of aurora kinase AURKA and the microtubule minus end-binding protein NUMA1 as well as a subset of AURKA targets which ensures proper spindle formation and timely chromosome alignment (PubMed:26246606)","3D-structure, Acetylation, Alternative splicing, Cytoplasm, Cytoskeleton, Disease variant, mRNA transport, Nuclear pore complex, Nucleus, Phosphoprotein, Protein transport, Proteomics identification, Reference proteome, Repeat, Translocation, Transport, WD repeat",aladin WD repeat nucleoporin [Source:HGNC Symbol;Acc:HGNC:13666]
3,AACS,Acetoacetyl-CoA synthetase,"Converts acetoacetate to acetoacetyl-CoA in the cytosol (By similarity). Ketone body-utilizing enzyme, responsible for the synthesis of cholesterol and fatty acids (By similarity)","Alternative splicing, ATP-binding, Cytoplasm, Fatty acid metabolism, Ligase, Lipid metabolism, Nucleotide-binding, Proteomics identification, Reference proteome",acetoacetyl-CoA synthetase [Source:HGNC Symbol;Acc:HGNC:21298]
4,AADACP1,NaN,NaN,NaN,arylacetamide deacetylase pseudogene 1 [Source:HGNC Symbol;Acc:HGNC:50305]
5,AAK1,AP2-associated protein kinase 1,"Regulates clathrin-mediated endocytosis by phosphorylating the AP2M1/mu2 subunit of the adaptor protein complex 2 (AP-2) which ensures high affinity binding of AP-2 to cargo membrane proteins during the initial stages of endocytosis (PubMed:11877457, PubMed:11877461, PubMed:12952931, PubMed:14617351, PubMed:17494869, PubMed:25653444). Isoform 1 and isoform 2 display similar levels of kinase activity towards AP2M1 (PubMed:17494869). Preferentially, may phosphorylate substrates on threonine residues (PubMed:11877457, PubMed:18657069). Regulates phosphorylation of other AP-2 subunits as well as AP-2 localization and AP-2-mediated internalization of ligand complexes (PubMed:12952931). Phosphorylates NUMB and regulates its cellular localization, promoting NUMB localization to endosomes (PubMed:18657069). Binds to and stabilizes the activated form of NOTCH1, increases its localization in endosomes and regulates its transcriptional activity (PubMed:21464124)","3D-structure, Acetylation, Alternative splicing, ATP-binding, Cell membrane, Cell projection, Coated pit, Endocytosis, Kinase, Membrane, Methylation, Nucleotide-binding, Pharmaceutical, Phosphoprotein, Proteomics identification, Reference proteome, Serine/threonine-protein kinase, Synapse, Transferase",AP2 associated kinase 1 [Source:HGNC Symbol;Acc:HGNC:19679]
6,AAMDC,Mth938 domain-containing protein,May play a role in preadipocyte differentiation and adipogenesis,"3D-structure, Alternative splicing, Cytoplasm, Proteomics identification, Reference proteome",adipogenesis associated Mth938 domain containing [Source:HGNC Symbol;Acc:HGNC:30205]
7,AAR2,Protein AAR2 homolog,Component of the U5 snRNP complex that is required for spliceosome assembly and for pre-mRNA splicing,"3D-structure, Acetylation, mRNA processing, mRNA splicing, Proteomics identification, Reference proteome, Spliceosome",AAR2 splicing factor [Source:HGNC Symbol;Acc:HGNC:15886]
8,AARS1,"Alanine--tRNA ligase, cytoplasmic","Catalyzes the attachment of alanine to tRNA(Ala) in a two-step reaction: alanine is first activated by ATP to form Ala-AMP and then transferred to the acceptor end of tRNA(Ala) (PubMed:27622773, PubMed:27911835, PubMed:28493438, PubMed:33909043). Also edits incorrectly charged tRNA(Ala) via its editing domain (PubMed:27622773, PubMed

In [6]:
gene_info[gene_info["gene"].str.contains("GRIA")]

,gene,protein_name,function,keywords,description
4007,GRIA1,Glutamate receptor 1,"Ionotropic glutamate receptor that functions as a ligand-gated cation channel, gated by L-glutamate and glutamatergic agonists such as alpha-amino-3-hydroxy-5-methyl-4-isoxazolepropionic acid (AMPA), quisqualic acid, and kainic acid (PubMed:1311100, PubMed:20805473, PubMed:21172611, PubMed:28628100, PubMed:35675825). L-glutamate acts as an excitatory neurotransmitter at many synapses in the central nervous system. Binding of the excitatory neurotransmitter L-glutamate induces a conformation change, leading to the opening of the cation channel, and thereby converts the chemical signal to an electrical impulse upon entry of monovalent and divalent cations such as sodium and calcium. The receptor then desensitizes rapidly and enters in a transient inactive state, characterized by the presence of bound agonist (By similarity). In the presence of CACNG2 or CACNG4 or CACNG7 or CACNG8, shows resensitization which is characterized by a delayed accumulation of current flux upon continued application of L-glutamate (PubMed:21172611). Resensitization is blocked by CNIH2 through interaction with CACNG8 in the CACNG8-containing AMPA receptors complex (PubMed:21172611). Calcium (Ca(2+)) permeability depends on subunits composition and, heteromeric channels containing edited GRIA2 subunit are calcium-impermeable. Also permeable to other divalents cations such as strontium(2+) and magnesium(2+) and monovalent cations such as potassium(1+) and lithium(1+) (By similarity)","3D-structure, Alternative splicing, Cell membrane, Cell projection, Disease variant, Disulfide bond, Endoplasmic reticulum, Endosome, Glycoprotein, Intellectual disability, Ion channel, Ion transport, Ligand-gated ion channel, Lipoprotein, Membrane, Palmitate, Phosphoprotein, Postsynaptic cell membrane, Proteomics identification, Receptor, Reference proteome, Signal, Synapse, Transmembrane, Transmembrane helix, Transport",glutamate ionotropic receptor AMPA type subunit 1 [Source:HGNC Symbol;Acc:HGNC:4571]
4008,GRIA2,Glutamate receptor 2,"Ionotropic glutamate receptor that functions as a ligand-gated cation channel, gated by L-glutamate and glutamatergic agonists such as alpha-amino-3-hydroxy-5-methyl-4-isoxazolepropionic acid (AMPA), quisqualic acid, and kainic acid (PubMed:20614889, PubMed:31300657, PubMed:8003671). L-glutamate acts as an excitatory neurotransmitter at many synapses in the central nervous system and plays an important role in fast excitatory synaptic transmission (PubMed:14687553). Binding of the excitatory neurotransmitter L-glutamate induces a conformation change, leading to the opening of the cation channel, and thereby converts the chemical signal to an electrical impulse upon entry of monovalent and divalent cations such as sodium and calcium (PubMed:20614889, PubMed:8003671). The receptor then desensitizes rapidly and enters in a transient inactive state, characterized by the presence of bound agonist (By similarity). In the presence of CACNG4 or CACNG7 or CACNG8, shows resensitization which is characterized by a delayed accumulation of current flux upon continued application of L-glutamate (By similarity). Through complex formation with NSG1, GRIP1 and STX12 controls the intracellular fate of AMPAR and the endosomal sorting of the GRIA2 subunit toward recycling and membrane targeting (By similarity)","3D-structure, Alternative splicing, Autism spectrum disorder, Cell membrane, Disease variant, Disulfide bond, Epilepsy, Glycoprotein, Intellectual disability, Ion channel, Ion transport, Ligand-gated ion channel, Lipoprotein, Membrane, Palmitate, Phosphoprotein, Postsynaptic cell membrane, Proteomics identification, Receptor, Reference proteome, RNA editing, Signal, Synapse, Transmembrane, Transmembrane helix, Transport, Ubl conjugation",glutamate ionotropic receptor AMPA type subunit 2 [Source:HGNC Symbol;Acc:HGNC:4572]
4009,GRIA3,Glutamate receptor 3,"Ionotropic glutamat

In [7]:
gene_info[gene_info["keywords"].str.contains("glut", case=False)].head(20)

,gene,protein_name,function,keywords,description
665,ASNS,Asparagine synthetase [glutamine-hydrolyzing],NaN,"3D-structure, Acetylation, Alternative splicing, Amino-acid biosynthesis, Asparagine biosynthesis, ATP-binding, Disease variant, Glutamine amidotransferase, Intellectual disability, Ligase, Nucleotide-binding, Phosphoprotein, Proteomics identification, Reference proteome",asparagine synthetase (glutamine-hydrolyzing) [Source:HGNC Symbol;Acc:HGNC:753]
724,ATP1B1,Sodium/potassium-transporting ATPase subunit beta-1,"This is the non-catalytic component of the active enzyme, which catalyzes the hydrolysis of ATP coupled with the exchange of Na(+) and K(+) ions across the plasma membrane. The beta subunit regulates, through assembly of alpha/beta heterodimers, the number of sodium pumps transported to the plasma membrane (PubMed:19694409). Plays a role in innate immunity by enhancing virus-triggered induction of interferons (IFNs) and interferon stimulated genes (ISGs). Mechanistically, enhances the ubiquitination of TRAF3 and TRAF6 as well as the phosphorylation of TAK1 and TBK1 (PubMed:34011520)","3D-structure, Alternative splicing, Cell adhesion, Cell membrane, Disulfide bond, Glutathionylation, Glycoprotein, Immunity, Innate immunity, Ion transport, Membrane, Phosphoprotein, Potassium, Potassium transport, Proteomics identification, Reference proteome, Signal-anchor, Sodium, Sodium transport, Sodium/potassium transport, Transmembrane, Transmembrane helix, Transport",ATPase Na+/K+ transporting subunit beta 1 [Source:HGNC Symbol;Acc:HGNC:804]
2004,CTPS1,CTP synthase 1,"CTP synthase involved in the de novo synthesis of CTP, a precursor of DNA, RNA and phospholipids (PubMed:16179339, PubMed:17189248, PubMed:17463002, PubMed:24870241, PubMed:28459447, PubMed:34583994). Catalyzes the ATP-dependent amination of UTP to CTP with either L-glutamine or ammonia as a source of nitrogen (PubMed:16179339, PubMed:24870241, PubMed:28459447, PubMed:34583994). CTPS1 CTP synthase activity plays a crucial role in the proliferation of activated lymphocytes and immunity; additional CTP being required to meet increased demand for DNA, RNA and lipid membrane biosynthesis in proliferating lymphocytes (PubMed:24870241, PubMed:8530356). In addition to CTP synthase activity, also acts as a protein deamidase that catalyzes the side chain deamidation of specific asparagine residues of proteins to aspartate (PubMed:40240600). Acts as a negative regulator of innate immunity by mediating deamidation of 'Asn-85' of IRF3, preventing IRF3 from binding DNA (By similarity). Facilitates chromatin relaxation in response to DNA damage by mediating deamidation of 'Asn-76' and 'Asn-77' of histone H1, thereby promoting subsequent acetylation of histone H1 at 'Lys-75' (H1K75ac), increasing chromatin accessibility to facilitate the recruitment of DNA repair proteins (PubMed:40240600)","3D-structure, Acetylation, Alternative splicing, ATP-binding, Chromatin regulator, Chromosome, Cytoplasm, DNA damage, Glutamine amidotransferase, Hydrolase, Immunity, Ligase, Nucleotide-binding, Nucleus, Phosphoprotein, Proteomics identification, Pyrimidine biosynthesis, Reference proteome",CTP synthase 1 [Source:HGNC Symbol;Acc:HGNC:2519]
3319,ETFA,"Electron transfer flavoprotein subunit alpha, mitochondrial","Heterodimeric electron transfer flavoprotein that accepts electrons from several mitochondrial dehydrogenases, including acyl-CoA dehydrogenases, glutaryl-CoA and sarcosine dehydrogenase (PubMed:10356313, PubMed:15159392, PubMed:15975918, PubMed:27499296, PubMed:9334218). It transfers the electrons to the main mitochondrial respiratory chain via ETF-ubiquinone oxidoreductase (ETF dehydrogenase) (PubMed:9334218). Required for normal mitochondrial fatty acid oxidation and normal amino acid metabolism (PubMed:12815589, PubMed:1430199, PubMed:1882842)","3D-structure, Acetylation, Alternative splicing, Direct protein sequencing, Disease variant, Electron transport, FAD, Flavoprotein, 

### ~~For each cell type, subset to top exons, and exons from genes with biologically interesting functions.~~
### Then, for exons compatible with multiple transcripts, select highest scoring transcript

In [ ]:
def score_transcript(x):
    tag = str(x['tag'])
    return (
        int("MANE_Select" in tag),
        int("MANE_Plus_Clinical" in tag),
        int("appris_principal" in tag),
        int("basic" in tag),
        int("CCDS" in tag),
        int("GENCODE_Primary" in tag),
        x['coding_nt_length'],    # prefer longer (more complete) CDS
        x['transcript'],         # deterministic alphabetical tiebreak
    ) 
    
transcript_list = []

for file in os.listdir("data/ctype_exons/annotated"):
    signif_exons = pd.read_csv(f"data/ctype_exons/annotated/{file}")
    signif_exons = signif_exons.rename(columns={signif_exons.columns[0]: "event"})
    
    # subset to protein coding trancripts, since we're interested in protein domains
    signif_coding_exons = signif_exons[signif_exons['transcript_type'] == "protein_coding"]
    
    # for each exon: keep the transcript with the highest score (based on GENCODE tags)
    signif_coding_exons['transcript_score'] = signif_coding_exons.apply(score_transcript, axis=1)
    idx = signif_coding_exons.groupby("event")['transcript_score'].idxmax()
    signif_coding_exons_max = signif_coding_exons.loc[idx]
    
    transcript_list.extend(signif_coding_exons_max['transcript'].tolist())

transcripts = list(set(transcript_list))  # unique transcripts 

### Get amino acid sequence for transcripts identified in previous step; save in FASTA format for input to InterProScan

In [11]:
proteins = Fasta("/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v49.pc_translations.fa")

# Build a dict keyed by ENST
protein_by_transcript = {}
for key in proteins.keys():
    parts = key.split("|")
    enst_versioned = parts[1].split(".")[0]
    protein_by_transcript[enst_versioned] = str(proteins[key])

In [ ]:
modified_transcript_products = {}
transcript_log = []

with open("data/proteins_filtered.fa", "w") as f:
    for key in proteins.keys():
        enst = key.split("|")[1].split(".")[0]
        if enst in transcripts:
            transcript_log.append(enst)
            seq = str(proteins[key]).rstrip("*")
            if "X" in seq:
                # Track where in the sequence the X was 
                modified_transcript_products[enst] = [i for i, c in enumerate(seq) if c == "X"]
                seq = seq.replace("X", "")
            f.write(f">{enst}\n{seq}\n")

In [13]:
len(transcript_log)

13946

In [14]:
len(modified_transcript_products)

416